# Kaggle eval runner

Runs the jobs too heavy for the laptop on Kaggle's free GPU, with `qwen2.5:7b` under Ollama. Same steps
as the Colab runner, built to run unattended.

**Before you start (right-hand panel):**
- Session options: Accelerator **GPU T4 x2**, Internet **on** (needs a phone-verified account).
- Add-ons > Secrets: add `GEMINI_API_KEY` and tick it for this notebook.
- Optional, to resume: add a dataset or earlier output holding a results zip. Any layout is found.

**Then: Save Version > Save & Run All.** You can close the tab. If a step fails, the steps that need it are
skipped and the last step still saves what exists to `/kaggle/working/kaggle_results_<time>.zip`.

**Keep this notebook private.** Its output holds your eval results.

In [ ]:
# Step 0: settings and helpers. Changes nothing on disk.
import json, os, re, shutil, subprocess, sys, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path

BRANCH = "stage-4-streaming-followups"
REPO_URL = "https://github.com/aryanwaingankar4/shipment-intelligence-rag.git"
OLLAMA_MODEL = "qwen2.5:7b"
GEMINI_MODEL = "gemini-3.5-flash-lite"  # saved answers are per model: changing this re-asks all of them
EVAL_RPM = "4"  # Gemini calls per minute; the free tier allows about 5
LAPTOP_DIGEST = "845dbda0ea48"  # the judge build behind the laptop's saved verdicts

# Kaggle saves everything in /kaggle/working as output, so only the final zip goes there.
ROOT = Path(os.environ.get("KAGGLE_RUNNER_ROOT", "/tmp/shipment-rag"))
KAGGLE = Path(os.environ.get("KAGGLE_RUNNER_KAGGLE", "/kaggle"))
INPUT, WORKING = KAGGLE / "input", KAGGLE / "working"
REPO = ROOT / "shipment-intelligence-rag"
VENV = ROOT / "venv"
LOGS = ROOT / "logs"
PY, CLI = VENV / "bin" / "python", VENV / "bin" / "shipment-rag"
OLLAMA_URL = "http://localhost:11434"
CACHE_PREFIX = "data/eval/.cache/"
CACHE_FILE = re.compile(r"[0-9a-f]{64}\.json")
CALIBRATION_MARK = LOGS / "calibration_passed"
FAILED = {}  # step number -> what went wrong, shown at the end and saved in run_info.json


def step(number, *needs):
    """Run a step body now. A failure is recorded, not raised, so Run All always reaches the zip."""
    def guarded(body):
        blocked = next((n for n in needs if n in FAILED), None)
        if blocked is not None:
            FAILED[number] = f"skipped, because Step {blocked} did not finish"
            print(f"Step {number} {FAILED[number]}.")
            return body
        try:
            body()
            FAILED.pop(number, None)
        except Exception as exc:
            FAILED[number] = f"{type(exc).__name__}: {exc}"
            print(f"Step {number} FAILED: {FAILED[number]}")
        return body
    return guarded


def gemini_key():
    """Only from Kaggle Secrets, and only handed to the jobs that need it. Never print it."""
    from kaggle_secrets import UserSecretsClient
    try:
        value = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception:
        value = None
    if not value:
        raise RuntimeError("No GEMINI_API_KEY: Add-ons > Secrets, add it and tick it for this notebook.")
    return value


def job_env(with_gemini=False):
    env = {**os.environ, "JUDGE_PROVIDER": "ollama", "JUDGE_MODEL": OLLAMA_MODEL,
           "CITATION_CHECKER": "nli", "GEMINI_MODEL": GEMINI_MODEL, "REWRITE_PROVIDER": "off",
           "OLLAMA_BASE_URL": OLLAMA_URL, "LLM_MAX_RETRIES": "1", "PYTHONUTF8": "1",
           "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
           "TOKENIZERS_PARALLELISM": "false"}
    # Nothing from the notebook's own Python may leak into the job's separate environment.
    for name in ("GEMINI_API_KEY", "PYTHONPATH", "PYTHONHOME", "MPLBACKEND"):
        env.pop(name, None)
    if with_gemini:
        env["GEMINI_API_KEY"] = gemini_key()
    return env


def run(cmd, log_name, env=None, show=True):
    """Run a command in the repo, keeping its output in logs/. Returns the exit code."""
    LOGS.mkdir(parents=True, exist_ok=True)
    proc = subprocess.Popen([str(c) for c in cmd], cwd=REPO, env=env or job_env(), text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            encoding="utf-8", errors="replace")
    with open(LOGS / log_name, "w", encoding="utf-8") as log:
        for line in proc.stdout:
            log.write(line)
            log.flush()
            if show:
                print(line, end="")
    return proc.wait()


def ollama_api(path, body=None, timeout=300):
    data = None if body is None else json.dumps(body).encode()
    request = urllib.request.Request(OLLAMA_URL + path, data=data,
                                     headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=timeout) as reply:
        return json.load(reply)


def ensure_ollama():
    """Start the Ollama server if it isn't answering. Safe to call any number of times."""
    try:
        ollama_api("/api/tags", timeout=3)
        return
    except OSError:
        pass
    if not shutil.which("ollama"):
        raise RuntimeError("Ollama is not installed: run Step 3 first.")
    LOGS.mkdir(parents=True, exist_ok=True)
    subprocess.Popen(["ollama", "serve"], stdout=open(LOGS / "ollama.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(30):
        time.sleep(1)
        try:
            ollama_api("/api/tags", timeout=3)
            return
        except OSError:
            continue
    raise RuntimeError("Ollama did not start: see logs/ollama.log")


def restore_cache(zip_path):
    """Copy saved answers and verdicts from a zip into the repo. Takes nothing else from it."""
    restored = 0
    base = (REPO / CACHE_PREFIX).resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for name in archive.namelist():
            if not name.startswith(CACHE_PREFIX) or name.endswith("/"):
                continue
            target = (REPO / name).resolve()
            if base not in target.parents:
                raise RuntimeError(f"unsafe path in zip: {name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))
            restored += 1
    return restored


def restore_from_input():
    """Bring back saved answers from everything attached, whatever the dataset's name or layout.

    Kaggle can unpack an uploaded zip, so this takes zips and loose cache files in any folder.
    Files are named by a hash of their request, so merging several sources can't mix answers up.
    """
    base = REPO / CACHE_PREFIX
    for folder, _, names in os.walk(INPUT, followlinks=True):
        for name in sorted(names):
            path = Path(folder) / name
            if name.endswith(".zip"):
                restore_cache(path)
            elif CACHE_FILE.fullmatch(name):
                target = base / name[:2] / name
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(path, target)
    return sum(1 for _ in base.rglob("*.json"))


print("Settings loaded. Gemini model:", GEMINI_MODEL)

In [ ]:
# Step 1: check for the GPU. If this fails: Settings > Accelerator > GPU T4 x2, then run again.
@step(1)
def check_gpu():
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    if gpu is None or gpu.returncode != 0:
        raise RuntimeError("No GPU on this session. Settings > Accelerator > GPU T4 x2.")
    if "P100" in gpu.stdout:  # current PyTorch builds may not run on it
        raise RuntimeError("This is the P100. Settings > Accelerator > GPU T4 x2.")
    print("GPU:", gpu.stdout.strip())

In [ ]:
# Step 2: get the code and install it in its own environment (about 5 minutes the first time).
# Its own environment avoids clashing with Kaggle's preinstalled packages. Skipped if already done.
@step(2)
def install_code():
    if not REPO.exists():
        ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO)],
                       check=True)
    if not CLI.exists():
        uv = [sys.executable, "-m", "uv"]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run([*uv, "venv", "--python", "3.12", str(VENV)], check=True)
        subprocess.run([*uv, "pip", "install", "--python", str(PY), "-e", str(REPO)], check=True)
    commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                            capture_output=True, text=True).stdout.strip()
    cuda = subprocess.run([str(PY), "-c", "import torch; print(torch.cuda.is_available())"],
                          capture_output=True, text=True, env=job_env()).stdout.strip()
    print(f"Code at {commit} on {BRANCH}. PyTorch sees the GPU: {cuda}")

In [ ]:
# Step 3: install Ollama, download the model and load it onto the GPU (about 5 minutes).
@step(3, 1)
def install_ollama():
    if not shutil.which("ollama"):
        subprocess.run(["apt-get", "update", "-qq"], check=True, stdout=subprocess.DEVNULL)
        subprocess.run(["apt-get", "install", "-y", "-qq", "zstd", "pciutils", "lshw"], check=True,
                       stdout=subprocess.DEVNULL)
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    ensure_ollama()
    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
    ollama_api("/api/generate", {"model": OLLAMA_MODEL, "prompt": "ok", "stream": False,
                                 "options": {"num_predict": 1}})
    digest = next(m["digest"][:12] for m in ollama_api("/api/tags")["models"]
                  if m["name"] == OLLAMA_MODEL)
    loaded = ollama_api("/api/ps")["models"][0]
    share = loaded.get("size_vram", 0) / max(loaded["size"], 1)
    print(f"{OLLAMA_MODEL} digest {digest}, {share:.0%} on the GPU")
    if share < 0.99:
        print("WARNING: not fully on the GPU, so this will be slow.")
    if digest != LAPTOP_DIGEST:
        print(f"NOTE: the laptop's verdicts came from digest {LAPTOP_DIGEST}, so they won't be reused.")

In [ ]:
# Step 4: bring back saved answers from anything attached, so Gemini isn't asked again.
# Nothing attached is fine: it just starts fresh. Answers saved under another model aren't reused.
@step(4, 2)
def restore_saved():
    found = restore_from_input()
    print(f"{found} saved files ready." if found else "Nothing attached: starting with no saved answers.")

In [ ]:
# Step 5: build the search index, and check retrieval matches the laptop. No LLM, no quota.
# Below the gate, the later steps are skipped: scores from a different index would mean nothing.
@step(5, 2)
def build_index():
    code = run([CLI, "ingest", "--rebuild"], "ingest.log", show=False)
    if code:
        raise RuntimeError("Indexing failed: see logs/ingest.log in the results zip")
    code = run([CLI, "eval"], "retrieval_eval.log")
    if code:
        raise RuntimeError("Retrieval is BELOW THE GATE: tell Claude.")
    print("\nRetrieval check: passed")

In [ ]:
# Step 6: paraphrase eval (D5). Does rewriting questions help retrieval? Local model only, no Gemini.
@step(6, 3, 5)
def paraphrase_eval():
    ensure_ollama()
    code = run([PY, "scripts/eval_rewrite.py", "--model", OLLAMA_MODEL,
                "--report", "data/eval/reports/stage4_rewrite.md"], "eval_rewrite.log")
    if code:
        raise RuntimeError(f"D5 exit {code}: see logs/eval_rewrite.log in the results zip")
    print("\nD5 finished.")

In [ ]:
# Step 7: does the judge still catch every planted flaw on this GPU? Local model only, no Gemini.
# Saved verdicts are ignored here on purpose: a GPU can answer differently from the laptop's CPU.
@step(7, 3, 5)
def calibrate():
    ensure_ollama()
    CALIBRATION_MARK.unlink(missing_ok=True)
    code = run([CLI, "eval-gen", "--calibration-only", "--no-cache"], "calibration.log")
    if code:
        raise RuntimeError("Calibration did not pass, so the golden set won't be scored. Tell Claude.")
    CALIBRATION_MARK.write_text(datetime.now(timezone.utc).isoformat())
    print("\nCalibration passed: Step 8 can run.")

In [ ]:
# Step 8: score the golden set. Uses Gemini for new answers and resumes from saved ones.
# Stops by itself on quota or repeated 503s, with everything so far kept.
# Verdicts go to the report, not this output, so the hand-label check stays blind.
@step(8, 4, 7)
def score_golden_set():
    ensure_ollama()
    if not CALIBRATION_MARK.exists():
        raise RuntimeError("Step 7 has not passed: the judge must pass calibration on this GPU.")
    code = run([CLI, "eval-gen", "--rpm", EVAL_RPM,
                "--report", "data/eval/reports/stage2_generation.md"],
               "eval_gen.log", env=job_env(with_gemini=True), show=False)
    log = (LOGS / "eval_gen.log").read_text(encoding="utf-8", errors="replace")
    report = REPO / "data/eval/reports/stage2_generation.md"
    partial = ""
    if report.exists():
        partial = next((l for l in report.read_text(encoding="utf-8").splitlines()
                        if "Partial run" in l), "")
    print("Exit code:", code)
    for line in log.splitlines():
        if line.startswith("Answer model"):
            print(line.split("; judge")[0])
    print("503 errors:", len(re.findall(r"ServerError: 503", log)),
          "| quota errors:", log.count("llm quota exhausted"))
    if code == 2:
        raise RuntimeError("Gemini wasn't reachable, so nothing was scored. Run the notebook again later.")
    if code:
        raise RuntimeError(f"Nothing was scored (exit {code}): see logs/eval_gen.log in the results zip.")
    if partial:
        print(partial.replace("*", "").strip("> "))
        print("Stopped early. To resume later, attach the results zip as an input and run all again.")
    else:
        print("All 26 questions scored.")

In [ ]:
# Step 9: prepare the blind hand-label file. Saved answers only: makes no Gemini calls.
# Holds cited sentences and their passages, with no checker verdicts.
@step(9, 5)
def prepare_labels():
    code = run([PY, "scripts/prepare_citation_labels.py", "--out",
                "data/eval/citation_labels_todo.json"],
               "prepare_labels.log", env=job_env(with_gemini=True), show=False)
    log = (LOGS / "prepare_labels.log").read_text(encoding="utf-8", errors="replace")
    print(*[l for l in log.splitlines() if l.startswith(("answers to label", "written to"))], sep="\n")
    if code:
        print(f"Exit {code}: no labelled answers yet, or see logs/prepare_labels.log.")

In [ ]:
# Step 10 (optional): list the Gemini flash models this key can use. Uses no generation quota.
# Only useful if the current model keeps returning 503. To switch, change GEMINI_MODEL in Step 0.
@step(10, 2)
def list_models():
    run([CLI, "models"], "models.log", env=job_env(with_gemini=True), show=False)
    names = (LOGS / "models.log").read_text(encoding="utf-8", errors="replace").splitlines()
    print(*[n for n in names if "flash" in n], sep="\n")

In [ ]:
# Step 11: save everything to one zip in /kaggle/working, check it holds no key, and report.
# Always runs, even after failed steps. Download the zip from the notebook's Output tab.
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
members = {}
for folder, prefix in [(REPO / "data/eval/.cache", CACHE_PREFIX),
                       (REPO / "data/eval/reports", "data/eval/reports/"),
                       (LOGS, "logs/")]:
    for path in sorted(folder.rglob("*")) if folder.exists() else []:
        if path.is_file() and path.name != "ollama.log":
            members[prefix + path.relative_to(folder).as_posix()] = path
labels = REPO / "data/eval/citation_labels_todo.json"
if labels.exists():
    members["data/eval/citation_labels_todo.json"] = labels

commit = subprocess.run(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
info = json.dumps({"finished_utc": stamp, "commit": commit, "branch": BRANCH,
                   "gemini_model": GEMINI_MODEL, "judge_model": OLLAMA_MODEL,
                   "machine": "Kaggle GPU", "failed_steps": FAILED}, indent=1)

try:
    key = gemini_key().encode()
except (ImportError, RuntimeError):
    key = b""  # no secret attached: the shape check below still runs
key_shaped = re.compile(rb"AIza[0-9A-Za-z_\-]{30,}")
contents = {name: path.read_bytes() for name, path in members.items()}
for name, data in {**contents, "run_info.json": info.encode()}.items():
    if (key and key in data) or key_shaped.search(data):
        raise RuntimeError(f"Not zipping: {name} contains an API key. Tell Claude. Nothing was saved.")

WORKING.mkdir(parents=True, exist_ok=True)
out = WORKING / f"kaggle_results_{stamp}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
    for name, data in contents.items():
        archive.writestr(name, data)
    archive.writestr("run_info.json", info)
print(f"{out}: {len(members)} files, {out.stat().st_size / 1024:.0f} KB")

try:
    ensure_ollama()
    ollama_api("/api/generate", {"model": OLLAMA_MODEL, "keep_alive": 0})
    print("Model unloaded.")
    if shutil.which("pkill"):  # a server left running could hold the saved version open
        subprocess.run(["pkill", "-x", "ollama"], check=False)
except (OSError, RuntimeError):
    pass  # no Ollama here, or it's already gone: nothing to unload

if FAILED:
    print("\nSteps that did not finish:")
    for number, why in sorted(FAILED.items()):
        print(f"  Step {number}: {why}")
else:
    print("\nAll steps finished.")